# 04 - Tier 2: the agent that uses tools

Tier 1 answered from memory and made numbers up. Tier 2 runs real queries.

**This is Abdelateef's Tier 2 design**, written in the same plain style as
the other notebooks so it plugs straight into the Tier 3 checker.

How it works:

1. **Plan** - the model lists the steps, each using one tool
2. **Run** - for every step we ask for just that one thing and run it
3. **Answer** - the model writes the answer from the rows that came back

Three ideas make it much harder to go wrong than a single prompt would:

| Idea | Why |
|---|---|
| The model never writes Python | it picks an operation, we do the sum |
| SQL is parsed, not word-matched | a column called `updated_at` contains "update" |
| Fields and filters read off the SQL | the model makes them up otherwise |

Everything it does is recorded in a log. Notebook 05 checks the answer
against that log.


## Setup

`sqlglot` is new - it parses SQL. `pip install sqlglot`


In [ ]:
import json
import re
import time

import duckdb
import sqlglot
from sqlglot import expressions as exp

DB = "../data/processed/evidenceiq.duckdb"
MODEL = "gemma3:4b"          # Abdelateef used gemma3:12b - use what you have
TOLERANCE = 0.01


## 1. Making the database safe

The agent writes its own SQL, so we have to stop it changing anything.
We parse the query into a tree and look at what it actually does, instead
of searching the text for banned words.


In [ ]:
# We never let the agent write to the database. Two safety nets: this check,
# and opening DuckDB with read_only=True.
#
# We parse the SQL into a syntax tree instead of searching for banned words.
# Word matching is easy to fool - a column called "updated_at" contains
# "update" - and it misses things a parser catches for free.

FORBIDDEN = {"insert", "update", "delete", "create", "drop", "alter", "copy",
             "command", "merge", "truncate", "attach", "detach", "install",
             "load", "pragma"}


def check_sql(sql):
    """Return None if the query is safe, otherwise a reason why it is not."""
    if not sql or not sql.strip():
        return "the query is empty"

    try:
        statements = sqlglot.parse(sql, read="duckdb")
    except Exception as e:
        return "could not parse the SQL: %s" % e

    if len(statements) != 1:
        return "only one statement is allowed"

    tree = statements[0]

    for node in tree.walk():
        key = getattr(node, "key", "").lower()
        if key in FORBIDDEN:
            return "forbidden operation: %s" % key

    if tree.find(exp.Select) is None:
        return "only SELECT queries are allowed"

    return None


In [ ]:
for sql in ["SELECT SUM(revenue) FROM sales",
            "DROP TABLE sales",
            "SELECT 1; DELETE FROM sales",
            "SELECT updated_at FROM sales",       # contains the word "update"
            "UPDATE sales SET revenue = 0"]:
    print("%-45s -> %s" % (sql[:45], check_sql(sql) or "safe"))


Note the fourth one. A word-matching check would have blocked it.


## 2. The log

`add_to_log` records a call, `numbers_in_log` pulls out every number the
tools returned, and `show_results` turns the log back into text for the
next prompt.


In [ ]:
def add_to_log(log, tool, code, ok, rows=None, error=None):
    """Every tool call goes in the log. The verifier only trusts what's here."""
    call = {"n": len(log), "tool": tool, "code": code,
            "ok": ok, "rows": rows or [], "error": error}
    log.append(call)
    return call


def numbers_in_log(log):
    """Every number the tools actually returned, with which call it came from."""
    found = []
    for call in log:
        if not call["ok"]:
            continue                      # a failed query proves nothing
        for row in call["rows"]:
            for value in row.values():
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    found.append((call["n"], float(value)))
    return found


def show_results(log, start=0):
    """Turn the log into text we can paste into the next prompt."""
    if len(log) <= start:
        return "(no queries were run)"
    text = ""
    for call in log[start:]:
        text += "[call %d] %s\n%s\n" % (call["n"], call["tool"], str(call["code"]).strip())
        text += "-> %s\n\n" % (json.dumps(call["rows"][:20]) if call["ok"]
                                else "FAILED: " + str(call["error"]))
    return text


## 3. Reading the query back

Two of the seven things we must show the user are the fields and filters
used. We read them off the SQL rather than asking the model.


In [ ]:
def sql_metadata(sql):
    """Which columns and filters did this query actually use?

    Two of the seven things we have to show the user are the fields and the
    filters. We read them off the query itself rather than asking the model,
    because the model will happily make them up.

    Aliases are skipped: in "SUM(revenue) AS total_revenue" the real field is
    revenue, not total_revenue.
    """
    try:
        tree = sqlglot.parse_one(sql, read="duckdb")
    except Exception:
        return [], []

    aliases = {a.alias for a in tree.find_all(exp.Alias) if a.alias}

    fields = []
    for column in tree.find_all(exp.Column):
        name = column.name
        if name and name not in aliases and name not in fields:
            fields.append(name)

    filters = []
    for where in tree.find_all(exp.Where):
        text = where.this.sql(dialect="duckdb")
        if text not in filters:
            filters.append(text)

    return fields, filters


In [ ]:
fields, filters = sql_metadata(
    "SELECT stock_code, ROUND(SUM(revenue),2) AS total FROM sales "
    "WHERE NOT is_cancellation AND year(invoice_date)=2011 GROUP BY stock_code")

print("fields :", fields)
print("filters:", filters)


`total` is missing from the fields on purpose - it is an alias we invented,
not a column in the data.


## 4. The tools


### run_sql


In [ ]:
def json_safe(value):
    """Database values that json.dumps cannot handle."""
    if hasattr(value, "isoformat"):          # dates and timestamps
        return value.isoformat()
    if hasattr(value, "item"):               # numpy numbers
        return value.item()
    return value


def run_sql(sql, log):
    """Run one SELECT and add the result to the log."""
    problem = check_sql(sql)
    if problem:
        return add_to_log(log, "run_sql", sql, False, error=problem)

    try:
        con = duckdb.connect(DB, read_only=True)
        cursor = con.execute(sql)
        columns = [d[0] for d in cursor.description]
        rows = [{c: json_safe(v) for c, v in zip(columns, row)}
                for row in cursor.fetchmany(50)]
        con.close()
        return add_to_log(log, "run_sql", sql, True, rows)
    except Exception as e:
        return add_to_log(log, "run_sql", sql, False, error=str(e))


In [ ]:
log = []

run_sql("SELECT ROUND(SUM(revenue),2) AS revenue FROM sales "
        "WHERE NOT is_cancellation AND year(invoice_date)=2011", log)

log[0]


That should be **9,809,614.01**, the same as q01 in our benchmark.


### run_python

The model does not write code. It picks an operation and hands us the
numbers, and we do the arithmetic. Safer, and easier to check later -
the operation and its inputs end up in the log.


In [ ]:
# The model does NOT get to write Python. It picks an operation from this
# list and gives us the numbers; we do the arithmetic ourselves.
#
# This is stricter than letting it write code, and it is also easier to check:
# the operation and its inputs are recorded in the log, so Tier 3 can redo the
# sum without trusting anything the model said.

OPERATIONS = ["pct_change", "difference", "ratio", "share", "sum", "mean"]


def calculate(operation, values):
    """Do the arithmetic. Returns (result, how_we_worked_it_out)."""
    if operation == "pct_change":
        if len(values) != 2:
            raise ValueError("pct_change needs [new, old]")
        new, old = values
        if old == 0:
            raise ValueError("cannot work out a percentage change from zero")
        return (new - old) / old * 100, "(%s - %s) / %s * 100" % (new, old, old)

    if operation == "difference":
        if len(values) != 2:
            raise ValueError("difference needs [a, b]")
        return values[0] - values[1], "%s - %s" % (values[0], values[1])

    if operation == "ratio":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("ratio needs [top, bottom] and bottom cannot be zero")
        return values[0] / values[1], "%s / %s" % (values[0], values[1])

    if operation == "share":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("share needs [part, total] and total cannot be zero")
        return values[0] / values[1] * 100, "%s / %s * 100" % (values[0], values[1])

    if operation == "sum":
        if not values:
            raise ValueError("sum needs at least one number")
        return sum(values), " + ".join(str(v) for v in values)

    if operation == "mean":
        if not values:
            raise ValueError("mean needs at least one number")
        return sum(values) / len(values), "mean of %d numbers" % len(values)

    raise ValueError("unknown operation: %s" % operation)


def run_python(request, log):
    """request = {"operation": ..., "values": [...], "result_name": ..., "unit": ...}"""
    operation = request.get("operation")
    values = [float(v) for v in (request.get("values") or [])]
    unit = request.get("unit") or ("%" if operation in ("pct_change", "share") else None)

    try:
        result, code = calculate(operation, values)
    except Exception as e:
        return add_to_log(log, "run_python", "%s(%s)" % (operation, values),
                          False, error=str(e))

    return add_to_log(log, "run_python", code, True,
                      [{"result_name": request.get("result_name") or operation,
                        "value": round(result, 4),
                        "unit": unit,
                        "operation": operation,
                        "inputs": values}])


In [ ]:
run_python({"operation": "pct_change",
            "values": [1503866.78, 1151263.73],
            "result_name": "nov_vs_oct"}, log)

log[-1]["rows"]


### make_chart


In [ ]:
CHART_TYPES = ["bar", "line", "scatter", "table", "none"]


def make_chart(spec, log):
    """We don't draw the chart, we just check the spec makes sense."""
    if spec.get("type") not in CHART_TYPES:
        return add_to_log(log, "make_chart", json.dumps(spec), False,
                          error="chart type must be one of %s" % CHART_TYPES)

    if spec.get("type") != "none":
        for needed in ("x", "y"):
            if not spec.get(needed):
                return add_to_log(log, "make_chart", json.dumps(spec), False,
                                  error="a chart needs both x and y")

    return add_to_log(log, "make_chart", json.dumps(spec), True, [spec])


## 5. Asking the model

We give Ollama a JSON schema, so the reply has to come back in the shape
we asked for. Much more reliable than asking nicely. If it still will not
parse, we try once more.


In [ ]:
def ask_gemma(system, question, shape):
    """Ask the model for JSON of a particular shape.

    `shape` is a JSON schema. Ollama forces the reply to match it, which is a
    lot more reliable than asking nicely and hoping. If the reply still will
    not parse we try once more with a blunter instruction.
    """
    import ollama

    for attempt in range(2):
        prompt = question if attempt == 0 else question + """

YOUR LAST REPLY WAS NOT VALID JSON.
Reply with ONE complete JSON object matching the schema. Keep the text short.
Do not write anything outside the JSON object."""

        reply = ollama.chat(model=MODEL, format=shape,
                            options={"temperature": 0, "num_predict": 2048},
                            messages=[{"role": "system", "content": system},
                                      {"role": "user", "content": prompt}])
        try:
            return json.loads(reply["message"]["content"])
        except Exception:
            continue

    return {"broken_json": True}


### The shapes we ask for

One shape per step. Small shapes work better - a 4B model fills in three
fields reliably and fifteen fields badly.


In [ ]:
# The shapes we ask the model to fill in. Keeping them small is deliberate -
# a 4B model fills in three fields reliably and fifteen fields badly.

PLAN_SHAPE = {
    "type": "object",
    "properties": {
        "sufficient_data": {"type": "boolean"},
        "reason": {"type": "string"},
        "steps": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "step": {"type": "integer"},
                "tool": {"type": "string", "enum": ["run_sql", "run_python", "make_chart"]},
                "objective": {"type": "string"},
            },
            "required": ["step", "tool", "objective"]}},
    },
    "required": ["sufficient_data", "steps"],
}

SQL_SHAPE = {"type": "object",
             "properties": {"sql": {"type": "string"}},
             "required": ["sql"]}

PYTHON_SHAPE = {
    "type": "object",
    "properties": {
        "operation": {"type": "string", "enum": OPERATIONS},
        "values": {"type": "array", "items": {"type": "number"}},
        "result_name": {"type": "string"},
        "unit": {"type": "string"},
    },
    "required": ["operation", "values", "result_name"],
}

CHART_SHAPE = {
    "type": "object",
    "properties": {
        "type": {"type": "string", "enum": CHART_TYPES},
        "source_tool_call": {"type": "integer"},
        "x": {"type": "string"},
        "y": {"type": "string"},
        "title": {"type": "string"},
    },
    "required": ["type"],
}

ANSWER_SHAPE = {
    "type": "object",
    "properties": {
        "findings": {"type": "string"},
        "claims": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "value": {"type": "number"},
                "unit": {"type": "string"},
                "from_call": {"type": "integer"},
                "calc": {"type": "string",
                         "enum": ["none", "pct_change", "share", "sum", "diff",
                                  "difference", "ratio", "mean"]},
                "inputs": {"type": "array", "items": {"type": "number"}},
            },
            "required": ["text"]}},
        "kpis": {"type": "object"},
        "limitations": {"type": "string"},
        "insufficient_data": {"type": "boolean"},
    },
    "required": ["findings", "claims"],
}


## 6. The prompts

The examples matter more than the rules. Gemma will not remember
"exclude cancellations" from a sentence, but it will copy a query it has
just been shown.


In [ ]:
SCHEMA = """
Database: a UK online gift wholesaler, Dec 2009 to 9 Dec 2011, 1,033,030 rows.

TABLE sales (one row per invoice line)
  invoice_no, stock_code, description, quantity, unit_price, customer_id,
  country, revenue (= quantity * unit_price), invoice_date, invoice_month,
  is_cancellation, is_product, is_outlier

TABLE dim_month   invoice_month, trading_days, net_revenue, gross_revenue,
                  is_complete_month
  net_revenue EXCLUDES cancellations (use this). gross_revenue includes them.
TABLE dim_product stock_code, description, units_sold, gross_revenue
TABLE dim_customer customer_id, country, first_order, last_order, orders, net_revenue

There is NO cost, profit, margin, discount, competitor or customer age data.
"""

RULES = """
RULES (the answer is wrong without these):
1. Revenue always excludes cancellations:  WHERE NOT is_cancellation
2. Product questions also need:            AND is_product AND NOT is_outlier
3. Group products by stock_code, and use mode(description) for the name.
4. For anything about months use dim_month (it has trading_days already).
5. December 2011 only has 8 trading days, so never compare it as a full month.
6. Units sold also needs quantity > 0.
7. Never say what CAUSED something, only what contributed to it.
8. If the data can't answer the question, say so instead of guessing.
"""

EXAMPLES = """
EXAMPLES

Total revenue in 2011:
  SELECT ROUND(SUM(revenue),2) AS revenue FROM sales
  WHERE NOT is_cancellation AND year(invoice_date) = 2011

Top 5 products in 2011:
  SELECT stock_code, mode(description) AS description, ROUND(SUM(revenue),2) AS revenue
  FROM sales
  WHERE NOT is_cancellation AND is_product AND NOT is_outlier
    AND year(invoice_date) = 2011
  GROUP BY stock_code ORDER BY revenue DESC LIMIT 5

Two months compared:
  SELECT invoice_month, ROUND(net_revenue,2) AS revenue, trading_days,
         is_complete_month
  FROM dim_month WHERE invoice_month IN ('2011-10','2011-11')
"""

SYSTEM = "You are a careful business data analyst.\n" + SCHEMA + RULES

PLAN_PROMPT = SYSTEM + """
Plan how to answer the question. Do NOT answer it - you have not seen any data.

List the steps in order. Each step uses one tool:
  run_sql      fetch numbers from the database
  run_python   work something out from numbers a query already returned
  make_chart   show the result

Set sufficient_data to false if the data cannot answer the question at all.
"""

ANSWER_PROMPT = SYSTEM + """
You asked for some queries and here are the results. Write the answer using
ONLY these numbers.

Every number in findings must also appear as a claim, with:
  from_call  which call it came from
  calc       none, unless you worked it out - then pct_change/share/sum/diff/ratio
  inputs     the numbers you worked it out from

Do not say what caused anything. Say what contributed.
"""


def sql_prompt(question, objective):
    return (SYSTEM + EXAMPLES
            + "\nQUESTION\n" + question
            + "\n\nWRITE ONE SELECT QUERY FOR THIS STEP ONLY\n" + objective)


def repair_prompt(question, objective, sql, error):
    return (SYSTEM + EXAMPLES
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nTHIS QUERY FAILED\n" + sql
            + "\n\nERROR\n" + str(error)
            + "\n\nWrite one corrected SELECT query.")


def python_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nNUMBERS THE QUERIES RETURNED\n" + show_results(log)
            + "\nPick the operation and give us the numbers. We do the arithmetic.")


def chart_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nRESULTS SO FAR\n" + show_results(log)
            + "\nChoose a chart. source_tool_call is the call number to plot.")


def answer_prompt(question, log, feedback=""):
    text = ("QUESTION\n" + question
            + "\n\nTOOL RESULTS\n" + show_results(log))
    if feedback:
        text += "\n\nYOUR LAST ANSWER FAILED CHECKING\n" + feedback
    return text


## 7. Two guards and a safety net

`touches_incomplete_month` stops us working out a month-over-month change
when one of the months is December 2011, which only has 8 trading days.

`claims_from_log` builds the claims ourselves if the model forgets to list
them. Without it, an answer with no claims would sail through Tier 3 with
nothing checked.


In [ ]:
def touches_incomplete_month(log):
    """Did any query return a month flagged as incomplete?"""
    for call in log:
        if not call["ok"]:
            continue
        for row in call["rows"]:
            if row.get("is_complete_month") is False:
                return True
    return False


def mentions_december_2011(question):
    text = question.lower()
    return ("december 2011" in text or "dec 2011" in text
            or "2011-12" in text)


def claims_from_log(log):
    """Build claims ourselves when the model forgets to.

    Tier 2 must never hand Tier 3 an answer with no claims - every number would
    then slip through unchecked. So if the model returns none, we make them from
    what the tools actually returned.
    """
    claims = []
    for call in log:
        if not call["ok"] or call["tool"] == "make_chart":
            continue
        for row in call["rows"]:
            for key, value in row.items():
                if isinstance(value, bool) or not isinstance(value, (int, float)):
                    continue
                claims.append({"text": "%s = %s" % (key, value),
                               "value": float(value),
                               "unit": row.get("unit"),
                               "from_call": call["n"],
                               "calc": "none",
                               "inputs": []})
    return claims


## 8. Tier 2


In [ ]:
def tier2(question, log=None, feedback=""):
    """Plan, run the steps, then write the answer from the real rows.

    The model never sees the database and never writes Python. It says what it
    wants; we run it and record what happened.
    """
    log = log if log is not None else []
    start = time.time()
    first_call = len(log)

    def stop(findings, limitations=None, insufficient=False, plan_steps=None):
        return {"question": question, "tier": 2, "findings": findings,
                "claims": [], "kpis": {}, "fields_used": [], "filters_used": [],
                "chart": None, "insufficient_data": insufficient,
                "limitations": limitations, "log": log, "retries": 0,
                "plan": plan_steps or [], "seconds": round(time.time() - start, 1)}

    # ---- 1. plan
    ask = question if not feedback else question + "\n\nLast attempt failed:\n" + feedback
    plan = ask_gemma(PLAN_PROMPT, ask, PLAN_SHAPE)

    if plan.get("broken_json"):
        return stop("The planner did not return valid JSON.",
                    "the model did not return valid JSON")

    steps = sorted(plan.get("steps") or [], key=lambda s: s.get("step", 0))
    plan_text = ["%s: %s" % (s.get("tool"), s.get("objective")) for s in steps]

    if not plan.get("sufficient_data", True):
        reason = plan.get("reason") or "The data cannot answer this question."
        return stop(reason, reason, True, plan_text)

    if not steps:
        return stop("The planner produced no steps to run.",
                    "the planner marked the question answerable but gave no steps",
                    False, plan_text)

    # ---- 2. run the steps
    chart = None

    for step in steps[:6]:
        tool = step.get("tool")
        objective = step.get("objective", "")

        if tool == "run_sql":
            request = ask_gemma(SYSTEM, sql_prompt(question, objective), SQL_SHAPE)
            call = run_sql(request.get("sql", ""), log)

            # one repair attempt, with the error message
            if not call["ok"]:
                fix = ask_gemma(SYSTEM,
                                repair_prompt(question, objective,
                                              call["code"], call["error"]),
                                SQL_SHAPE)
                call = run_sql(fix.get("sql", ""), log)

            if not call["ok"]:
                return stop("The query could not be made to run.",
                            call["error"], False, plan_text)

        elif tool == "run_python":
            # Do not work out a month-over-month change when one of the months
            # is incomplete. December 2011 has 8 trading days.
            if touches_incomplete_month(log) or mentions_december_2011(question):
                continue
            request = ask_gemma(SYSTEM, python_prompt(question, objective, log),
                                PYTHON_SHAPE)
            run_python(request, log)

        elif tool == "make_chart":
            spec = ask_gemma(SYSTEM, chart_prompt(question, objective, log),
                             CHART_SHAPE)
            call = make_chart(spec, log)
            if call["ok"] and spec.get("type") != "none":
                chart = spec

    # ---- 3. refuse if nothing worked
    useful = [c for c in log[first_call:]
              if c["ok"] and c["tool"] in ("run_sql", "run_python")]
    if not useful:
        return stop("No query produced any evidence, so there is no answer to give.",
                    "no successful query or calculation", False, plan_text)

    # ---- 4. write the answer from the real rows
    reply = ask_gemma(ANSWER_PROMPT, answer_prompt(question, log, feedback),
                      ANSWER_SHAPE)

    if reply.get("broken_json"):
        return stop("The model did not return a usable answer.",
                    "the model did not return valid JSON", False, plan_text)

    claims = reply.get("claims") or []
    if not claims:
        claims = claims_from_log(log[first_call:])

    # ---- 5. fields and filters come from the SQL, not from the model
    fields, filters = [], []
    for call in log[first_call:]:
        if call["tool"] != "run_sql" or not call["ok"]:
            continue
        f, w = sql_metadata(call["code"])
        fields += [x for x in f if x not in fields]
        filters += [x for x in w if x not in filters]

    limitations = reply.get("limitations")
    if (touches_incomplete_month(log) or mentions_december_2011(question)) \
            and not limitations:
        limitations = ("December 2011 has only 8 trading days, so it is not "
                       "comparable with a complete month.")

    return {"question": question, "tier": 2,
            "findings": reply.get("findings", ""),
            "claims": claims,
            "kpis": reply.get("kpis") or {},
            "fields_used": fields or [],
            "filters_used": filters or [],
            "chart": chart,
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": limitations,
            "log": log, "retries": 0, "plan": plan_text,
            "seconds": round(time.time() - start, 1)}


## 9. Run it

Make sure Ollama is running: `ollama pull gemma3:4b`


In [ ]:
answer = tier2('What was our total revenue in 2011?')

print('PLAN:')
for step in answer['plan']:
    print(' -', step)

print()
print('FINDINGS:', answer['findings'])
print('FIELDS  :', answer['fields_used'])
print('FILTERS :', answer['filters_used'])


### What did it actually run?


In [ ]:
for call in answer['log']:
    print('[%d] %s   ok=%s' % (call['n'], call['tool'], call['ok']))
    print(call['code'])
    print('->', call['rows'] if call['ok'] else call['error'])
    print()


### The claims

Every number listed separately, with where it came from. Notebook 05
needs this - you cannot reliably pull numbers out of a paragraph.


In [ ]:
for c in answer['claims']:
    print(c.get('value'), c.get('unit'), '|', c.get('text'))
    print('    from call', c.get('from_call'), '| calc:', c.get('calc'))


## 10. Does it refuse impossible questions?

There is no cost data, so profit margin cannot be worked out.


In [ ]:
bad_q = tier2('What was our profit margin in 2011?')

print('refused?    ', bad_q['insufficient_data'])
print('queries run:', len(bad_q['log']))
print('says:       ', bad_q['findings'])


## 11. The harder ones


In [ ]:
for q in ['Did November 2011 beat October 2011 on revenue, and by what percentage?',
          'List the five products that generated the most revenue in 2011, with the revenue for each.']:
    print('=' * 70)
    print(q)
    print('=' * 70)
    a = tier2(q)
    print(a['findings'])
    print()


## What we found

Write this down, it goes in the report:

- did it remember `NOT is_cancellation`?
- did any query fail, and did the repair attempt fix it?
- did it fill in `calc` and `inputs` for calculated numbers?

**When Gemma writes a bad query, fix the prompt, not the code.** Add another
worked example to `EXAMPLES`.

The numbers are real now. But nothing checks that the *sentence* matches
the rows - that is notebook 05.
